# Fermionic (dyon) field transition — **L=4**, CNN

Sweeps a **fermionic perturbation** $H = H_{TC} - h_f\sum S_e$, $S_e = X_a\,Z_b$ (the
L-shaped two-body operator from the figure) at **L=4, OBC**, $h_x=h_y=h_z=0$.

**Physics.** $Z$ moves an $e$ (charge, on vertices; anticommutes with $A_v=XXXX$); $X$ moves an
$m$ (flux, on plaquettes; anticommutes with $B_p=ZZZZ$). $S_e=X_aZ_b$ **binds** an $e$ and an $m$
into the composite **fermion** $\varepsilon=e\times m$ and hops it. This is *not* an equal $X+Z$
single-site field (that would condense $e$ and $m$ independently) — the product proliferates the
*bound* dyon.

**Sign structure (important).** $X\cdot Z$ has no $\sigma^y$ so $H$ is **real**, but it is **not
sign-problem-free**: in the $Z$ basis $X_aZ_b$ has sign-indefinite off-diagonals, and small-$L$ ED
shows the exact ground state is **signful** (~46% negative amplitudes for any $h_f>0$). That sign
structure *is* the fermionic statistics of $\varepsilon$. A positive-only `float64` amplitude cannot
represent it, so **$h_f>0$ runs use the complex Combo CNN** (`config.py` auto-sets `dtype=complex`
for $h_f\neq0$, same path as the $h_y$ cut); the $h_f=0$ anchor stays the cheap real CNN.

**Structure.** One cell per network: each trains at a single $h_f$ and `main.py` saves the
`.mpack` weights + a `.json` with the full observable set (energy, V-score, $\langle S_e\rangle$,
$\langle A_v\rangle$, $\langle B_p\rangle$, Rényi-2). The final cell collects everything and plots
vs $h_f$. Everything lands under Google Drive so it survives Colab disconnects (skip-if-complete ⇒
just re-run to resume). **Runtime:** the complex path pays a large one-off XLA compile (~15–20 min)
*per point* (each point is a fresh process), plus cheap stepping (~1.2 s/step); so ~20–25 min/point
⇒ the 11-point sweep spans **~2–3 Colab sessions** — run a few cells, let Drive persist them, re-run
to resume. (The $h_f=0$ anchor is the fast real CNN.)

**Locating the transition.** Watch $\langle S_e\rangle$ rise $0\to$ finite (peak in $d\langle S_e\rangle/dh_f$),
$\langle A_v\rangle$ & $\langle B_p\rangle$ degrade from 1, and a Rényi-2 feature.


## Before you start
1. **Runtime → Change runtime type → GPU** (A100/L4 preferred; T4 works but slow).
2. Have your **GitHub token** (private repo) ready.
3. Run top-to-bottom. **After the install cell you may need to restart the runtime** — then resume
   from *Mount Google Drive*.
4. The **$h_f=0.0$ cell runs first** as an anchor: it must reproduce the pure toric code
   ($E\approx-25.0$, $\langle A_v\rangle\approx\langle B_p\rangle\approx1$, $\langle S_e\rangle\approx0$).


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install the pinned stack

`requirements.txt` pins a **CPU** `jaxlib`; on Colab we install the **CUDA 12** build of the same
`jax==0.5.2` instead, plus `netket==3.16.1.post1`, `flax==0.10.4`.

⚠️ If pip changes the `jaxlib` already loaded in this session, Colab will ask you to **restart the
runtime** — do it, then resume from the *Mount Google Drive* cell (do NOT re-run this install cell).

In [ ]:
%pip install -q "jax[cuda12]==0.5.2" "jaxlib==0.5.1" netket==3.16.1.post1 flax==0.10.4
print("\nInstall finished. If a 'restart runtime' notice appeared, do Runtime > Restart session,")
print("then continue from the 'Mount Google Drive' cell (do NOT re-run this install cell).")

In [ ]:
import jax, jaxlib, netket, flax
print("jax", jax.__version__, "| jaxlib", jaxlib.__version__,
      "| netket", netket.__version__, "| flax", flax.__version__)
print("devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), \
    "No GPU visible to JAX -- set Runtime>GPU and/or re-check the jax[cuda12] install."
print("OK: GPU visible.")

## 3. Mount Google Drive  *(resume point after any runtime restart)*
Results and the cloned repo live under `MyDrive/2D-TC-colab/` so they survive disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Clone (or update) the repo into Drive
Private repo → paste a GitHub token when prompted. It is used only for the clone, then stripped from
the stored remote (not saved to Drive). Branch: **`fermionic-perturbation`**.

In [ ]:
import os, getpass, subprocess

DRIVE_ROOT = "/content/drive/MyDrive/2D-TC-colab"   # persists across sessions
BRANCH     = "fermionic-perturbation"                # has the fermionic-field code
REPO_DIR   = os.path.join(DRIVE_ROOT, "2D-TC")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    tok = getpass.getpass("GitHub token (repo read access): ")
    url = f"https://{tok}@github.com/SanzharBissenali/2D-TC.git"
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    "https://github.com/SanzharBissenali/2D-TC.git"], check=True)
    del tok, url
else:
    print("repo already present; fetching + switching to", BRANCH)
    # An existing clone may be a shallow single-branch clone (e.g. another notebook
    # cloned --depth 1 --branch <other>), so `git checkout BRANCH` would find no local
    # ref. Widen the refspec, fetch just our branch, then hard-switch to its remote tip.
    subprocess.run(["git", "-C", REPO_DIR, "config", "remote.origin.fetch",
                    "+refs/heads/*:refs/remotes/origin/*"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "--depth", "1", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

os.chdir(REPO_DIR)
head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "| branch:", head)
assert "_generate_fermion_pairs" in open("model/geometry.py").read(), \
    "fermionic-field code missing -- wrong branch?"
print("OK: fermionic-field code present.")

## 5. Sweep configuration
L=4 Combo-small CNN (identical ansatz to the $h_z$/$h_y$ cuts): custom sampler, `dt=0.01`,
`diag_shift=6e-5`, `sim_time=3.5` (350 TDVP steps). `H_F_LIST` = $\{0.0, 0.1, \dots, 1.0\}$.
`run_hf(hf)` shells out to `main.py` (which saves `.mpack` + `.json`) and is **skip-if-complete**.
No dtype flag is passed: `config.py` auto-selects `float64` at $h_f=0$ and **`complex`** at $h_f>0$
(the signful path).

In [ ]:
import os, subprocess, json

REPO   = os.getcwd()
OUTDIR = os.path.join(REPO, "results", "nqs")
os.makedirs(OUTDIR, exist_ok=True)

LX       = 4
SEED     = 0
H_F_LIST = [round(0.1 * k, 2) for k in range(11)]   # 0.0 .. 1.0

# Combo-small CNN, same as the hz/hy baseline. dtype is auto-selected by config.py:
# float64 (positive) at h_f=0, complex (signful) at h_f>0 -- no dtype flag needed here.
BASE_FLAGS = (
    "--outindex 1 "
    "--architecture Combo --channels_noninv 1,16 --channels_inv 16,8,1 --kernel_size 2 "
    "--n_samples_fin 8192 --use_custom_sampler "
    "--dt 0.01 --sim_time 3.5 "
    f"--seed {SEED}"
)

def _jobid(hf):  return f"L{LX}_hx0.00_hz0.00_hf{hf:.2f}"
def _base(hf):   return os.path.join(OUTDIR, f"G-equiv_1_{_jobid(hf)}")

def _is_complete(hf):
    b = _base(hf)
    if not os.path.exists(b + ".mpack"):
        return False
    try:  # NaN/early-diverged runs also write .mpack, so require the Se observable too
        d = json.load(open(b + ".json"))
        return len(d.get("order_params", {}).get("Se_mean", [])) > 0
    except Exception:
        return False

def run_hf(hf, diag_shift=6e-5):
    """Train one network at a single h_f; main.py saves weights (.mpack) + observables (.json)."""
    jobid = _jobid(hf)
    if _is_complete(hf):
        print(f"== skip {jobid} (already complete) =="); return
    print(f"\n{'='*70}\n RUN {jobid}   (diag_shift={diag_shift})\n{'='*70}", flush=True)
    cmd = (
        f"cd {OUTDIR} && PYTHONPATH={REPO} python {REPO}/main.py "
        f"--jobid {jobid} --Lx {LX} --hx 0.0 --hy 0.0 --hz 0.0 --h_f {hf:.4f} "
        f"--diag_shift {diag_shift} {BASE_FLAGS}"
    )
    rc = subprocess.run(["bash", "-lc", cmd], env=dict(os.environ, PYTHONPATH=REPO)).returncode
    print(f"===== {jobid} exit {rc} =====", flush=True)

print(f"{len(H_F_LIST)} points:", H_F_LIST)

## 6. Train — one network per cell
Each cell trains a single $h_f$ and writes `results/nqs/G-equiv_1_L4_hx0.00_hz0.00_hf<hf>.{json,mpack}`.
Run them in order; re-running a completed cell is a no-op (skip-if-complete).

**If a high-$h_f$ point diverges** (V-score stuck near 1, energy NaN — the SR instability seen past the
$h_y$ transition), re-run that cell with a larger shift, e.g. `run_hf(0.9, diag_shift=1e-3)`.

In [ ]:
run_hf(0.0)   # anchor: expect E~-25.0, <A_v>~<B_p>~1, <S_e>~0

In [ ]:
run_hf(0.1)

In [ ]:
run_hf(0.2)

In [ ]:
run_hf(0.3)

In [ ]:
run_hf(0.4)

In [ ]:
run_hf(0.5)

In [ ]:
run_hf(0.6)

In [ ]:
run_hf(0.7)

In [ ]:
run_hf(0.8)

In [ ]:
run_hf(0.9)

In [ ]:
run_hf(1.0)

## 7. Collect all results & plot
Re-globs the per-run JSONs from disk (survives restarts), tabulates the final observables, plots
$E$, $\langle S_e\rangle$ (+ susceptibility), the stabilizers $\langle A_v\rangle$/$\langle B_p\rangle$,
and Rényi-2 vs $h_f$, and saves a summary JSON + figure.

In [ ]:
import os, json
import numpy as np, matplotlib.pyplot as plt

REPO   = os.getcwd()
OUTDIR = os.path.join(REPO, "results", "nqs")
LX       = 4
H_F_LIST = [round(0.1 * k, 2) for k in range(11)]
TAIL     = 10

def _c(x):
    return complex(x) if not isinstance(x, dict) else complex(x.get("real", np.nan), x.get("imag", 0.0))
def _last(lst):
    return float(lst[-1]) if lst else np.nan

rows = []
for hf in H_F_LIST:
    p = os.path.join(OUTDIR, f"G-equiv_1_L{LX}_hx0.00_hz0.00_hf{hf:.2f}.json")
    if not os.path.exists(p):
        rows.append({"hf": hf, "pending": True}); continue
    d  = json.load(open(p))
    e  = [_c(x).real for x in d.get("energy", [])]
    v  = [_c(x).real for x in d.get("Vscore", [])]
    op = d.get("order_params", {})
    ren = op.get("renyi2_entropy", [])
    rows.append({
        "hf": hf,
        "E":  (float(np.median(e[-TAIL:])) if e else np.nan),
        "V":  (float(np.median(v[-TAIL:])) if v else np.nan),
        "Se": _last(op.get("Se_mean", [])),
        "Se_std": _last(op.get("Se_std", [])),
        "Bp": _last(op.get("Bp_mean", [])),
        "Av": _last(op.get("Av_mean", [])),
        "S2": (float(ren[-1][0]) if ren else np.nan),
    })

print(f"{'h_f':>5} {'E':>10} {'Vscore':>9} {'<S_e>':>8} {'<B_p>':>7} {'<A_v>':>7} {'S2':>7}")
for r in rows:
    if r.get("pending"):
        print(f"{r['hf']:>5.2f}  (pending)"); continue
    print(f"{r['hf']:>5.2f} {r['E']:>10.4f} {r['V']:>9.1e} {r['Se']:>8.4f} {r['Bp']:>7.3f} {r['Av']:>7.3f} {r['S2']:>7.3f}")

done = [r for r in rows if not r.get("pending") and np.isfinite(r["E"])]
hf = np.array([r["hf"] for r in done]); E = np.array([r["E"] for r in done])
Se = np.array([r["Se"] for r in done]); Bp = np.array([r["Bp"] for r in done])
Av = np.array([r["Av"] for r in done]); S2 = np.array([r["S2"] for r in done])

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax[0, 0].plot(hf, E, "o-"); ax[0, 0].set(title="Energy", xlabel="$h_f$", ylabel="E")
ax[0, 1].plot(hf, Se, "o-", color="crimson")
ax[0, 1].set(title=r"$\langle S_e\rangle$ (fermion order parameter)", xlabel="$h_f$",
             ylabel=r"$\langle S_e\rangle$")
if len(hf) > 2:
    ax2 = ax[0, 1].twinx()
    ax2.plot(hf, np.gradient(Se, hf), "s--", color="gray", alpha=0.6)
    ax2.set_ylabel(r"$d\langle S_e\rangle/dh_f$")
ax[1, 0].plot(hf, Bp, "o-", label=r"$\langle B_p\rangle$ (m)")
ax[1, 0].plot(hf, Av, "s-", label=r"$\langle A_v\rangle$ (e)")
ax[1, 0].set(title="Stabilizers", xlabel="$h_f$", ylabel="stabilizer"); ax[1, 0].legend()
ax[1, 1].plot(hf, S2, "o-", color="purple")
ax[1, 1].set(title=r"R\'enyi-2 entropy $S_2$", xlabel="$h_f$", ylabel="$S_2$")
fig.tight_layout()

figdir = os.path.join(REPO, "figures"); os.makedirs(figdir, exist_ok=True)
fig.savefig(os.path.join(figdir, "fermionic_transition_L4.png"), dpi=140)
with open(os.path.join(REPO, "results", "fermionic_L4_summary.json"), "w") as f:
    json.dump(rows, f, indent=2)
print("\nsaved: figures/fermionic_transition_L4.png  +  results/fermionic_L4_summary.json")
plt.show()